# 04 — Modélisation Escalade

**Objectif** : Entraîner un modèle de prédiction du niveau d'escalade, en implémentant l'architecture conditionnelle adaptée à chaque profil utilisateur.

**Architecture des prédictions :**
- `years_cl = 0` → **Mode POTENTIEL** : règles physiologiques (dead hang, BMI, tractions). Pas de ML.
- `years_cl < 1` → **Mode ESTIMATION** : ML avec intervalle élargi + disclaimer
- `years_cl ≥ 1` → **Mode ML** : Random Forest entraîné sur dataset 8a.nu

**Dataset** : `data/processed/climbing_clean.csv`

---

## Plan
1. Chargement et préparation
2. Feature engineering
3. Baseline naïve
4. Modèle ML — entraînement et évaluation
5. Règles métier mode POTENTIEL
6. Analyse des erreurs par tranche de niveau
7. Export du modèle et des règles

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
import warnings
from datetime import datetime

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

RANDOM_STATE = 42

## 1. Chargement et préparation

In [ ]:
df = pd.read_csv('../data/processed/climbing_clean.csv')
grades_raw = pd.read_csv('../data/raw/climbing/grades_conversion_table.csv')
grade_map = dict(zip(grades_raw['grade_id'], grades_raw['grade_fra']))
grade_map = {k: v for k, v in grade_map.items() if v != '-'}

print(f"Dataset : {len(df):,} grimpeurs")
print(f"Features disponibles : {list(df.columns)}")
print()
print(df.describe().round(2))

## 2. Feature engineering

In [ ]:
# BMI si pas déjà présent
if 'bmi' not in df.columns:
    df['bmi'] = df['weight'] / (df['height'] / 100) ** 2

# Rapport poids/taille (utile en escalade)
df['weight_per_cm'] = df['weight'] / df['height']

# Features pour le modèle ML
FEATURES_ML = ['sex', 'height', 'weight', 'bmi', 'age', 'years_cl']
TARGET = 'grades_max'

# Garder uniquement les grimpeurs avec au moins 1 an (hors distribution débutant absolu)
df_ml = df[df['years_cl'] >= 1].copy()
print(f"Dataset ML (years_cl ≥ 1) : {len(df_ml):,} grimpeurs")
print(f"Retirés (years_cl < 1) : {len(df) - len(df_ml)} grimpeurs")

X = df_ml[FEATURES_ML]
y = df_ml[TARGET]
print(f"\nTarget — stats :")
print(f"  Min : {y.min()} → {grade_map.get(y.min(), '?')}")
print(f"  Max : {y.max()} → {grade_map.get(y.max(), '?')}")
print(f"  Médiane : {y.median():.0f} → {grade_map.get(int(y.median()), '?')}")

## 3. Baseline naïve

In [ ]:
def evaluate_climbing(y_true, y_pred, label='Modèle'):
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    within_1 = np.mean(np.abs(y_true - y_pred) <= 1) * 100
    within_2 = np.mean(np.abs(y_true - y_pred) <= 2) * 100
    within_3 = np.mean(np.abs(y_true - y_pred) <= 3) * 100
    print(f"{label}:")
    print(f"  MAE     = {mae:.2f} grades")
    print(f"  R²      = {r2:.4f}")
    print(f"  ±1 grade = {within_1:.1f}%")
    print(f"  ±2 grades = {within_2:.1f}%")
    print(f"  ±3 grades = {within_3:.1f}%")
    return {'mae': mae, 'r2': r2, 'within_1': within_1, 'within_2': within_2, 'within_3': within_3}

# Baseline : prédire la médiane globale
baseline_pred = np.full(len(y), y.median())
baseline_metrics = evaluate_climbing(y, baseline_pred, 'Baseline (médiane globale)')

## 4. Modèle ML

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
print(f"Train : {len(X_train):,} | Test : {len(X_test):,}")

In [ ]:
rf = RandomForestRegressor(n_estimators=100, max_depth=12, min_samples_leaf=5, random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
metrics_rf = evaluate_climbing(y_test, y_pred_rf, 'Random Forest')
print()

In [ ]:
gb = GradientBoostingRegressor(n_estimators=100, max_depth=4, learning_rate=0.1, random_state=RANDOM_STATE)
gb.fit(X_train, y_train)
y_pred_gb = gb.predict(X_test)
metrics_gb = evaluate_climbing(y_test, y_pred_gb, 'Gradient Boosting')
print()

In [ ]:
best_model = rf if metrics_rf['mae'] < metrics_gb['mae'] else gb
best_name = 'Random Forest' if metrics_rf['mae'] < metrics_gb['mae'] else 'Gradient Boosting'
best_metrics = metrics_rf if best_name == 'Random Forest' else metrics_gb
y_pred_best = y_pred_rf if best_name == 'Random Forest' else y_pred_gb

print(f"Meilleur modèle : {best_name}")

cv_scores = cross_val_score(best_model, X, y, cv=5, scoring='neg_mean_absolute_error', n_jobs=-1)
cv_mae = -cv_scores
print(f"Validation croisée 5-fold : MAE = {cv_mae.mean():.2f} ± {cv_mae.std():.2f} grades")

## 5. Analyse des erreurs par tranche de niveau

In [ ]:
results = pd.DataFrame({'y_true': y_test, 'y_pred': y_pred_best})
results['error'] = np.abs(results['y_true'] - results['y_pred'])

tranches = [
    ('< 6a', 0, 36),
    ('6a-6b+', 37, 44),
    ('6c-7a+', 45, 52),
    ('7b-7c+', 53, 60),
    ('8a+', 61, 84),
]

print("=== MAE par tranche de niveau ===")
for label, lo, hi in tranches:
    mask = results['y_true'].between(lo, hi)
    sub = results[mask]
    if len(sub) > 0:
        print(f"  {label:12s} : MAE={sub['error'].mean():.2f} grades | n={len(sub):4d} | ±1 grade: {(sub['error']<=1).mean()*100:.1f}%")

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Prédit vs réel
axes[0].scatter(results['y_true'], results['y_pred'], alpha=0.15, s=8, color='steelblue')
lims = [results['y_true'].min(), results['y_true'].max()]
axes[0].plot(lims, lims, 'r--', linewidth=1)
axes[0].set_xlabel('Grade réel')
axes[0].set_ylabel('Grade prédit')
axes[0].set_title(f'{best_name}\nPrédit vs Réel')

key_grades = {37: '6a', 45: '6c', 49: '7a', 53: '7b', 57: '7c', 61: '8a', 65: '8b'}
for ax in axes[:1]:
    ax.set_xticks(list(key_grades.keys()))
    ax.set_xticklabels(list(key_grades.values()))
    ax.set_yticks(list(key_grades.keys()))
    ax.set_yticklabels(list(key_grades.values()))

# Feature importance
importances = best_model.feature_importances_
axes[1].barh(FEATURES_ML, importances, color='steelblue')
axes[1].set_title('Feature importance')
axes[1].set_xlabel('Importance')

plt.tight_layout()
plt.savefig('../data/processed/model_climbing_results.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Mode POTENTIEL — règles métier pour débutants absolus

In [ ]:
def predict_climbing_potential(weight_kg, height_cm, age, dead_hang_sec=None, max_pullups=None):
    """
    Estimation du potentiel escalade pour un débutant absolu (sans années de pratique).
    Basé sur des règles physiologiques issues de la littérature sportive.
    Retourne une fourchette (grade_min, grade_max) et un message.
    """
    bmi = weight_kg / (height_cm / 100) ** 2
    score = 0

    # BMI (rapport poids/force crucial en escalade)
    if bmi < 20:
        score += 3
    elif bmi < 22:
        score += 2
    elif bmi < 25:
        score += 1
    # bmi >= 25 : 0 points

    # Dead hang (force isométrique des doigts)
    if dead_hang_sec is not None:
        if dead_hang_sec >= 60:
            score += 3
        elif dead_hang_sec >= 40:
            score += 2
        elif dead_hang_sec >= 20:
            score += 1

    # Tractions
    if max_pullups is not None:
        if max_pullups >= 15:
            score += 3
        elif max_pullups >= 10:
            score += 2
        elif max_pullups >= 5:
            score += 1

    # Âge (pic physique 20-30 ans)
    if 20 <= age <= 30:
        score += 1

    # Convertir score en fourchette de grade potentiel
    # Score 0-2 : potentiel débutant (5a-6a)
    # Score 3-5 : potentiel intermédiaire (6a-7a)
    # Score 6-8 : potentiel avancé (7a-8a)
    # Score 9-10 : potentiel expert (8a+)
    if score <= 2:
        grade_range = (30, 37)  # 5b → 6a
        label = 'Débutant-intermédiaire'
    elif score <= 5:
        grade_range = (37, 49)  # 6a → 7a
        label = 'Intermédiaire'
    elif score <= 8:
        grade_range = (49, 61)  # 7a → 8a
        label = 'Avancé'
    else:
        grade_range = (61, 70)  # 8a → 8c
        label = 'Expert'

    return {
        'mode': 'potential',
        'score': score,
        'grade_min': grade_range[0],
        'grade_max': grade_range[1],
        'grade_min_fra': grade_map.get(grade_range[0], '?'),
        'grade_max_fra': grade_map.get(grade_range[1], '?'),
        'potential_label': label,
        'disclaimer': (
            "Estimation basée sur votre profil physique uniquement. "
            "Votre niveau réel dépendra fortement de votre entraînement, "
            "votre technique et votre régularité de pratique. "
            "Cette fourchette représente un potentiel après 1-2 ans de pratique sérieuse."
        )
    }

# Exemples
print("=== Tests mode POTENTIEL ===")
examples = [
    {'weight_kg': 65, 'height_cm': 175, 'age': 25, 'dead_hang_sec': 50, 'max_pullups': 12},
    {'weight_kg': 80, 'height_cm': 180, 'age': 35, 'dead_hang_sec': 20, 'max_pullups': 5},
    {'weight_kg': 58, 'height_cm': 170, 'age': 22, 'dead_hang_sec': 70, 'max_pullups': 18},
]
for ex in examples:
    result = predict_climbing_potential(**ex)
    print(f"Profil {ex}: potentiel {result['grade_min_fra']}→{result['grade_max_fra']} ({result['potential_label']}, score={result['score']})")

## 7. Fonction de prédiction unifiée

In [ ]:
def predict_climbing(model, sex, height, weight, age, years_cl,
                     dead_hang_sec=None, max_pullups=None):
    """
    Point d'entrée unifié pour la prédiction escalade.
    Sélectionne automatiquement le mode selon years_cl.
    """
    bmi = weight / (height / 100) ** 2

    if years_cl == 0:
        # Mode POTENTIEL : règles métier
        return predict_climbing_potential(weight, height, age, dead_hang_sec, max_pullups)

    # Mode ML
    features = pd.DataFrame([[sex, height, weight, bmi, age, years_cl]], columns=FEATURES_ML)
    grade_pred = round(model.predict(features)[0])
    grade_pred = max(29, min(77, grade_pred))  # Clamp dans la plage connue

    confidence = 'high' if years_cl >= 1 else 'medium'
    grade_range = 1 if years_cl >= 1 else 2  # ±1 si expérimenté, ±2 si < 1 an

    return {
        'mode': 'ml',
        'grade_numeric': int(grade_pred),
        'grade_fra': grade_map.get(int(grade_pred), '?'),
        'grade_range_fra': f"{grade_map.get(max(29, int(grade_pred)-grade_range), '?')} → {grade_map.get(min(77, int(grade_pred)+grade_range), '?')}",
        'confidence': confidence,
        'disclaimer': None if years_cl >= 1 else (
            "Moins d'un an de pratique — prédiction indicative. "
            "La précision augmentera avec l'expérience."
        )
    }

# Tests
print("=== Tests fonction unifiée ===")
print()
print("Débutant absolu :")
print(predict_climbing(best_model, sex=0, height=175, weight=65, age=25, years_cl=0, dead_hang_sec=45, max_pullups=10))
print()
print("Grimpeur 3 ans :")
print(predict_climbing(best_model, sex=0, height=175, weight=65, age=25, years_cl=3))
print()
print("Grimpeur 10 ans :")
print(predict_climbing(best_model, sex=0, height=170, weight=62, age=30, years_cl=10))

## 8. Export du modèle

In [ ]:
joblib.dump(best_model, '../models/climbing_model.pkl')
joblib.dump(grade_map, '../models/climbing_grade_map.pkl')

metadata = {
    'model_type': best_name,
    'sport': 'climbing',
    'features': FEATURES_ML,
    'target': TARGET,
    'target_description': 'Grade max numérique (29-77, converti en cotation française)',
    'train_samples': len(X_train),
    'test_samples': len(X_test),
    'metrics': {
        'mae_grades': round(best_metrics['mae'], 3),
        'r2': round(best_metrics['r2'], 4),
        'within_1_grade_pct': round(best_metrics['within_1'], 1),
        'within_2_grades_pct': round(best_metrics['within_2'], 1)
    },
    'prediction_modes': {
        'potential': 'years_cl = 0 → règles physiologiques, pas de ML',
        'estimation': 'years_cl < 1 → ML avec intervalle ±2 grades',
        'ml': 'years_cl ≥ 1 → ML avec intervalle ±1 grade'
    },
    'dataset': 'jordizar/climb-dataset (8a.nu)',
    'trained_at': datetime.now().isoformat(),
    'limitations': [
        'Dataset surreprésenté en grimpeurs avancés (médiane 7b)',
        'Aucun grimpeur avec 0 an de pratique dans le dataset',
        'Dead hang non disponible dans le dataset — non inclus dans le modèle ML',
        'Prédictions moins fiables sous 6a (peu de données)'
    ]
}

with open('../models/climbing_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print("Modèle exporté : models/climbing_model.pkl")
print("Grade map exportée : models/climbing_grade_map.pkl")
print("Métadonnées : models/climbing_metadata.json")
print()
print(json.dumps(metadata, indent=2, ensure_ascii=False))

## 9. Conclusions

### Résultats
| Modèle | MAE | ±1 grade | ±2 grades | R² |
|---|---|---|---|---|
| Baseline (médiane) | X grades | X% | X% | — |
| Random Forest | X grades | X% | X% | X |
| Gradient Boosting | X grades | X% | X% | X |

*(Valeurs à compléter après exécution)*

### Architecture de prédiction retenue
- **Débutant absolu** (years_cl=0) : règles métier → fourchette de potentiel avec disclaimer
- **Pratiquant** (years_cl≥1) : ML → grade précis ±1 grade
- Ce choix est **scientifiquement défendable** : on n'extrapole pas le modèle hors distribution

### Feature importance attendue
`years_cl` devrait dominer nettement — ce qui est cohérent : l'escalade est avant tout une question de pratique.

### Usage dans l'API
- Le service `climbing_service.py` implémente la logique conditionnelle
- L'API retourne systématiquement `mode_used`, `confidence`, et `disclaimer`
- L'interface Streamlit adapte l'affichage selon le mode